In [1]:
import numpy as np

import src.util as util
import src.svm as svm

In [2]:
def get_words(message):
    """Get the normalized list of words from a message string.

    This function should split a message into words, normalize them, and return
    the resulting list. For splitting, you should split on spaces. For normalization,
    you should convert everything to lowercase.

    Args:
        message: A string containing an SMS message

    Returns:
       The list of normalized words from the message.
    """

    # *** START CODE HERE ***
    return message.lower().split()
    # *** END CODE HERE ***


def create_dictionary(messages):
    """Create a dictionary mapping words to integer indices.

    This function should create a dictionary of word to indices using the provided
    training messages. Use get_words to process each message.

    Rare words are often not useful for modeling. Please only add words to the dictionary
    if they occur in at least five messages.

    Args:
        messages: A list of strings containing SMS messages

    Returns:
        A python dict mapping words to integers.
    """

    # *** START CODE HERE ***
    first_idx = {}   
    counts = {}     

    for msg in messages:
        for w in get_words(msg):
            if w not in first_idx:
                first_idx[w] = len(first_idx) + 1
            counts[w] = counts.get(w, 0) + 1

    result = {w: [first_idx[w], c] for w, c in counts.items() if c >= 5}

    sorted_words = sorted(result.items(), key=lambda x: x[1][0])
    reindexed = {w: [new_i, c] for new_i, (w, (_, c)) in enumerate(sorted_words, start=0)}

    result = {w: reindexed[w][0] for w in reindexed.keys()}
    return result

def transform_text(messages, word_dictionary):
    """Transform a list of text messages into a numpy array for further processing.

    This function should create a numpy array that contains the number of times each word
    appears in each message. Each row in the resulting array should correspond to each
    message and each column should correspond to a word.

    Use the provided word dictionary to map words to column indices. Ignore words that
    are not present in the dictionary. Use get_words to get the words for a message.

    Args:
        messages: A list of strings where each string is an SMS message.
        word_dictionary: A python dict mapping words to integers.

    Returns:
        A numpy array marking the words present in each message.
    """
    # *** START CODE HERE ***
    m = len(messages)
    n = len(word_dictionary)
    x=np.zeros((m, n))
    for i, j in enumerate(messages):
        for word in get_words(j):
            if word in word_dictionary:
                x[i,word_dictionary[word] - 1] = x[i,word_dictionary[word] - 1] + 1
    return x  


In [3]:
def get_words(message):
    """Get the normalized list of words from a message string.

    This function should split a message into words, normalize them, and return
    the resulting list. For splitting, you should split on spaces. For normalization,
    you should convert everything to lowercase.

    Args:
        message: A string containing an SMS message

    Returns:
       The list of normalized words from the message.
    """

    # *** START CODE HERE ***
    return message.lower().split()
    # *** END CODE HERE ***


def create_dictionary(messages):
    """Create a dictionary mapping words to integer indices.

    This function should create a dictionary of word to indices using the provided
    training messages. Use get_words to process each message.

    Rare words are often not useful for modeling. Please only add words to the dictionary
    if they occur in at least five messages.

    Args:
        messages: A list of strings containing SMS messages

    Returns:
        A python dict mapping words to integers.
    """

    # *** START CODE HERE ***
    first_idx = {}   
    counts = {}     

    for msg in messages:
        for w in get_words(msg):
            if w not in first_idx:
                first_idx[w] = len(first_idx) + 1 
            counts[w] = counts.get(w, 0) + 1

    result = {w: [first_idx[w], c] for w, c in counts.items() if c >= 5}

    sorted_words = sorted(result.items(), key=lambda x: x[1][0])
    reindexed = {w: [new_i, c] for new_i, (w, (_, c)) in enumerate(sorted_words, start=0)}

    result = {w: reindexed[w][0] for w in reindexed.keys()}
    return result

    # *** END CODE HERE ***


def transform_text(messages, word_dictionary):
    """Transform a list of text messages into a numpy array for further processing.

    This function should create a numpy array that contains the number of times each word
    appears in each message. Each row in the resulting array should correspond to each
    message and each column should correspond to a word.

    Use the provided word dictionary to map words to column indices. Ignore words that
    are not present in the dictionary. Use get_words to get the words for a message.

    Args:
        messages: A list of strings where each string is an SMS message.
        word_dictionary: A python dict mapping words to integers.

    Returns:
        A numpy array marking the words present in each message.
    """
    # *** START CODE HERE ***
    m = len(messages)
    n = len(word_dictionary)
    x=np.zeros((m, n))
    for i, j in enumerate(messages):
        for word in get_words(j):
            if word in word_dictionary:
                x[i,word_dictionary[word]] = x[i,word_dictionary[word]] + 1
    return x 
    # *** END CODE HERE ***


def fit_naive_bayes_model(matrix, labels):
    """Fit a naive bayes model.

    This function should fit a Naive Bayes model given a training matrix and labels.

    The function should return the state of that model.

    Feel free to use whatever datatype you wish for the state of the model.

    Args:
        matrix: A numpy array containing word counts for the training data
        labels: The binary (0 or 1) labels for that training data

    Returns: The trained model
    """

    # *** START CODE HERE ***
    _, n = matrix.shape
    phi_y = np.mean(labels)
    phi_k_y1 = (1 + matrix[labels == 1].sum(axis=0))/(n + matrix[labels == 1].sum())
    phi_k_y0 = (1 + matrix[labels == 0].sum(axis=0))/(n + matrix[labels == 0].sum())
    return phi_y, phi_k_y1, phi_k_y0
    # *** END CODE HERE ***

For predictions we take into account de following. We say that our prediction is class 1 if
$$
p(y=1|x) > p(y=0|x).
$$
This, taking into account Bayes is,
$$
p(x|y=1)p(y=1) > p(x|y=0)p(y=0).
$$
If we take log on both sides,
$$
\log p(x|y=1)p(y=1) > \log p(x|y=0)p(y=0),
$$
$$
\log p(x|y=1)+\log p(y=1) > \log p(x|y=0)+\log p(y=0),
$$

In [4]:
def predict_from_naive_bayes_model(model, matrix):
    """Use a Naive Bayes model to compute predictions for a target matrix.

    This function should be able to predict on the models that fit_naive_bayes_model
    outputs.

    Args:
        model: A trained model from fit_naive_bayes_model
        matrix: A numpy array containing word counts

    Returns: A numpy array containg the predictions from the model
    """
    # *** START CODE HERE ***
    phi_y, phi_k_y1, phi_k_y0 = model

    #return matrix @ (np.log(phi_k_y1) - np.log(phi_k_y0)) + np.log(phi_y / (1 - phi_y)) >= 0
    is_1 = matrix @ np.log(phi_k_y1) + np.log(phi_y)
    is_0 = matrix @ np.log(phi_k_y0) + (np.log(1- phi_y))

    prediction = np.array([1 if is_1[i] > is_0[i] else 0 for i in range(len(is_1))])
    return prediction

    # *** END CODE HERE ***

In [5]:
def get_top_five_naive_bayes_words(model, dictionary):
    """Compute the top five words that are most indicative of the spam (i.e positive) class.

    Ues the metric given in 6c as a measure of how indicative a word is.
    Return the words in sorted form, with the most indicative word first.

    Args:
        model: The Naive Bayes model returned from fit_naive_bayes_model
        dictionary: A mapping of word to integer ids

    Returns: The top five most indicative words in sorted order with the most indicative first
    """
    # *** START CODE HERE ***
    phi_y, phi_k_y1, phi_k_y0 = model
    prob_word = np.zeros(len(dictionary))
    for i, j in enumerate(dictionary):
        probs = np.zeros(len(dictionary))
        probs[i] = 1
        prob_word[i] = np.log((probs@phi_k_y1)/(probs@phi_k_y0))


    keys = list(dictionary.keys())
    words_to_return = []
    for i in np.argsort(prob_word)[::-1][0:5]:
        words_to_return.append(keys[i])
    return words_to_return
    # *** END CODE HERE ***


def compute_best_svm_radius(
    train_matrix, train_labels, val_matrix, val_labels, radius_to_consider
):
    """Compute the optimal SVM radius using the provided training and evaluation datasets.

    You should only consider radius values within the radius_to_consider list.
    You should use accuracy as a metric for comparing the different radius values.

    Args:
        train_matrix: The word counts for the training data
        train_labels: The spma or not spam labels for the training data
        val_matrix: The word counts for the validation data
        val_labels: The spam or not spam labels for the validation data
        radius_to_consider: The radius values to consider

    Returns:
        The best radius which maximizes SVM accuracy.
    """
    # *** START CODE HERE ***
    best_radius = radius_to_consider[0]
    best_accuracy = 0
    for radius in radius_to_consider:
        predictions = svm.train_and_predict_svm(train_matrix, train_labels, val_matrix, radius)
        accuracy = np.mean(predictions == val_labels)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_radius = radius
    return best_radius

    # *** END CODE HERE ***


def main():
    train_messages, train_labels = util.load_spam_dataset(r"C:\cs229-journey\cs229-journey\problem_sets\PS2\data\ds6_train.tsv")
    val_messages, val_labels = util.load_spam_dataset(r"C:\cs229-journey\cs229-journey\problem_sets\PS2\data\ds6_val.tsv")
    test_messages, test_labels = util.load_spam_dataset(r"C:\cs229-journey\cs229-journey\problem_sets\PS2\data\ds6_test.tsv")

    dictionary = create_dictionary(train_messages)

    #util.write_json("./output/p06_dictionary", dictionary)

    train_matrix = transform_text(train_messages, dictionary)

    #np.savetxt("./output/p06_sample_train_matrix", train_matrix[:100, :])

    val_matrix = transform_text(val_messages, dictionary)
    test_matrix = transform_text(test_messages, dictionary)

    naive_bayes_model = fit_naive_bayes_model(train_matrix, train_labels)

    naive_bayes_predictions = predict_from_naive_bayes_model(
        naive_bayes_model, test_matrix
    )

    #np.savetxt("./output/p06_naive_bayes_predictions", naive_bayes_predictions)

    naive_bayes_accuracy = np.mean(naive_bayes_predictions == test_labels)

    print(
        "Naive Bayes had an accuracy of {} on the testing set".format(
            naive_bayes_accuracy
        )
    )

    top_5_words = get_top_five_naive_bayes_words(naive_bayes_model, dictionary)

    print("The top 5 indicative words for Naive Bayes are: ", top_5_words)

    #util.write_json("./output/p06_top_indicative_words", top_5_words)

    optimal_radius = compute_best_svm_radius(
        train_matrix, train_labels, val_matrix, val_labels, [0.01, 0.1, 1, 10]
    )

    #util.write_json("./output/p06_optimal_radius", optimal_radius)

    print("The optimal SVM radius was {}".format(optimal_radius))

    svm_predictions = svm.train_and_predict_svm(
        train_matrix, train_labels, test_matrix, optimal_radius
    )

    svm_accuracy = np.mean(svm_predictions == test_labels)

    print(
        "The SVM model had an accuracy of {} on the testing set".format(
            svm_accuracy, )
    )


main()

Naive Bayes had an accuracy of 0.978494623655914 on the testing set
The top 5 indicative words for Naive Bayes are:  ['claim', 'won', 'prize', 'tone', 'urgent!']
The optimal SVM radius was 0.1
The SVM model had an accuracy of 0.9695340501792115 on the testing set
